In [1]:
from datasets import load_dataset
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

## load dataset

In [2]:
ds = load_dataset("BothBosu/scam-dialogue")
print(ds)

README.md: 0.00B [00:00, ?B/s]

scam-dialogue_train.csv: 0.00B [00:00, ?B/s]

scam-dialogue_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1280 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/320 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 1280
    })
    test: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 320
    })
})


In [3]:
# convert to pandas for easier manipulation
df = pd.DataFrame(ds['train'])
print(f"shape: {df.shape}")
df.head()

shape: (1280, 3)


,dialogue,type,label
0,"caller: Hello, is this John? receiver: Yes, it...",ssn,1
1,"caller: Hello, is this John? receiver: Yeah, t...",ssn,1
2,"caller: Hello, is this Mr. Johnson? receiver: ...",ssn,1
3,"caller: Hello, is this John? receiver: Yeah, t...",ssn,1
4,"caller: Hello, this is Officer Johnson from th...",ssn,1


In [4]:
# check what columns we have
df.columns

Index(['dialogue', 'type', 'label'], dtype='object')

In [5]:
# check label distribution
if 'label' in df.columns:
    print(df['label'].value_counts())
else:
    print("available columns:", df.columns.tolist())

label
1    640
0    640
Name: count, dtype: int64


## extract features

In [6]:
def count_urgency_words(text):
    urgency_words = ['immediately', 'urgent', 'now', 'today', 'expires', 'hurry', 
                     'asap', 'quickly', 'deadline', 'limited time']
    text_lower = str(text).lower()
    return sum(1 for word in urgency_words if word in text_lower)

def count_verification_requests(text):
    verification_patterns = [
        r'verify\s+your', r'confirm\s+your', r'provide\s+your',
        r'enter\s+your', r'social\s+security', r'account\s+number',
        r'password', r'credit\s+card', r'bank\s+account'
    ]
    return sum(1 for pattern in verification_patterns if re.search(pattern, str(text).lower()))

In [7]:
# figure out which column has the text
# common names: 'text', 'dialogue', 'conversation', 'message'
text_col = None
for col in ['text', 'dialogue', 'conversation', 'message', 'content']:
    if col in df.columns:
        text_col = col
        break

if text_col:
    print(f"using column: {text_col}")
    df['urgency_score'] = df[text_col].apply(count_urgency_words)
    df['verification_requests'] = df[text_col].apply(count_verification_requests)
else:
    print("need to identify text column manually:")
    print(df.columns.tolist())

using column: dialogue


## train baseline model

In [8]:
# update these based on actual column names
label_col = 'label'  # change if different

X = df[text_col]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train: {len(X_train)}, test: {len(X_test)}")

train: 1024, test: 256


In [9]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=1000,
    ngram_range=(1, 2)
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_baseline = LogisticRegression(max_iter=1000, random_state=42)
model_baseline.fit(X_train_tfidf, y_train)

y_pred_baseline = model_baseline.predict(X_test_tfidf)
accuracy_baseline = (y_pred_baseline == y_test).mean()

print(f"baseline accuracy: {accuracy_baseline:.2%}")

baseline accuracy: 100.00%


## enhanced model with custom features

In [10]:
feature_cols = ['urgency_score', 'verification_requests']
df_train = df.loc[X_train.index]
df_test = df.loc[X_test.index]

custom_features_train = df_train[feature_cols].values
custom_features_test = df_test[feature_cols].values

scaler = StandardScaler()
custom_features_train_scaled = scaler.fit_transform(custom_features_train)
custom_features_test_scaled = scaler.transform(custom_features_test)

X_train_combined = hstack([X_train_tfidf, custom_features_train_scaled])
X_test_combined = hstack([X_test_tfidf, custom_features_test_scaled])

model_enhanced = LogisticRegression(max_iter=1000, random_state=42)
model_enhanced.fit(X_train_combined, y_train)

y_pred_enhanced = model_enhanced.predict(X_test_combined)
accuracy_enhanced = (y_pred_enhanced == y_test).mean()

print(f"enhanced accuracy: {accuracy_enhanced:.2%}")
print(f"improvement: {(accuracy_enhanced - accuracy_baseline)*100:.2f} points")

enhanced accuracy: 100.00%
improvement: 0.00 points


## results

In [11]:
print(classification_report(y_test, y_pred_enhanced))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       128
           1       1.00      1.00      1.00       128

    accuracy                           1.00       256
   macro avg       1.00      1.00      1.00       256
weighted avg       1.00      1.00      1.00       256



In [12]:
cm = confusion_matrix(y_test, y_pred_enhanced)
print(cm)

[[128   0]
 [  0 128]]


## feature importance

In [13]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model_enhanced.coef_[0][:-2]

scam_indices = np.argsort(coefficients)[-10:]
print("top scam words:")
for idx in reversed(scam_indices):
    print(f"  {feature_names[idx]}: {coefficients[idx]:.3f}")

top scam words:
  computer: 2.501
  refund: 1.972
  virus: 1.776
  gift: 1.678
  gift card: 1.628
  need: 1.536
  windows: 1.082
  microsoft: 0.974
  uh: 0.911
  card: 0.847


In [14]:
legit_indices = np.argsort(coefficients)[:10]
print("top legit words:")
for idx in legit_indices:
    print(f"  {feature_names[idx]}: {coefficients[idx]:.3f}")

top legit words:
  package: -1.589
  insurance: -1.554
  delivery: -1.191
  business: -1.078
  offering: -1.051
  caller hi: -1.034
  hi: -0.987
  xyz: -0.981
  alright: -0.931
  calling xyz: -0.893


In [15]:
custom_coefs = model_enhanced.coef_[0][-2:]
print("custom features:")
print(f"  urgency: {custom_coefs[0]:.3f}")
print(f"  verification requests: {custom_coefs[1]:.3f}")

custom features:
  urgency: -0.095
  verification requests: 1.315


In [16]:
print(f"total samples: {len(df)}")

total samples: 1280
